![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 02: Prompt Engineering and Retrieval-Augmented Generation)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 2A: Prompt Foundations

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Main output</td><td>A specification-driven prompt builder and an in-context learning experiment with a conflicting-demonstration case</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m02a-overview)
2. [Setup and Background](#m02a-setup)
3. [Core Concepts](#m02a-core-concepts)
4. [Guided Implementation](#m02a-guided-implementation)
5. [Testing and Analysis](#m02a-testing)
6. [Student Tasks](#m02a-student-tasks)
7. [Submission and Reflection](#m02a-submission)

---

<a id="m02a-overview"></a>

### 1. Overview and Learning Goals

This session locates runtime prompting relative to model-level control. In `M01B` you separated the model layer from the workflow layer of an agentic system. This session zooms into the boundary between them. A deployed language model arrives with behaviour that was fixed during training: pretraining decides what it knows and how it writes, instruction tuning decides how it interprets task requests, and alignment decides which requests it refuses or reshapes. None of that can be changed at call time. What you can change at call time is the prompt, the context you supply, and the evidence that retrieval places into that context. Prompt engineering is the discipline of using this runtime lever well.

A useful analogy is hiring a highly trained professional. You cannot re-run their university degree for each task, but you can change the brief you hand them. A vague brief produces a plausible but unpredictable result; a precise brief with the task, the source material, the constraints, a worked example and the required deliverable format produces a checkable result. This session teaches you to write prompts as explicit specifications rather than as conversational wishes.

The second theme is in-context learning. A frozen model can infer a task from a description alone (zero-shot), from one demonstration (one-shot), or from several demonstrations (few-shot). Demonstrations are not decoration: they define the task convention the model will follow. You will verify this by deliberately supplying conflicting demonstrations and observing that the model learns the flipped convention from the prompt, even when it contradicts common sense. This experiment matters for agentic AI because later workflows will assemble demonstrations and retrieved evidence automatically, and a workflow that assembles misleading context will mislead the model.

By the end of this lab, you should be able to distinguish model-level control (pretraining, instruction tuning, alignment) from runtime control (prompt, context, retrieval); write a prompt as an explicit specification with a task, context, constraints, examples and an output contract, adding a role only when it genuinely matters; build prompts programmatically so that specification fields can be varied one at a time; run zero-, one- and few-shot experiments on the same task; and explain what a conflicting-demonstration experiment reveals about how models use context. These skills carry directly into [M02B-Prompt-Engineering-Control-Loop](M02B-Prompt-Engineering-Control-Loop.ipynb), the Flowise prompt work in [M03B-Flowise-Chatbot-Prompt-Memory](../../M03-Context-Orchestration/Flowise/M03B-Flowise-Chatbot-Prompt-Memory.md) and the retrieval work in [M02C-Retrieval-Augmented-Generation](M02C-Retrieval-Augmented-Generation.ipynb).

<a id="m02a-setup"></a>

### 2. Setup and Background

This is the first session in the unit that calls a real large language model. We use the Google Gemini API because it offers a free tier suitable for coursework and because later Flowise practicals in M03 use the same provider. The API key is loaded with the same safe pattern introduced in `M01A`: the key comes from an environment variable or a hidden `getpass` prompt, it is never written into the notebook source, and the notebook never prints it.

You do not need an API key to complete this lab. Every model call in this notebook goes through one helper function, `generate_text`, which works in two modes. In **live mode**, the helper sends your prompt to the Gemini API and returns the model response. In **offline mode**, it returns a recorded example output that was captured from a real instruction-tuned model when this lab was authored. The recorded outputs are deterministic, so offline mode always behaves the same way; live mode is probabilistic, so your outputs may differ slightly from the interpretations written in this notebook. Both modes teach the same lesson, because the lesson is about how prompts shape behaviour, not about any single response.

To obtain a free Gemini API key, sign in at [Google AI Studio](https://aistudio.google.com), create an API key, and keep it private. In Colab, the cleanest option is to store it as a Colab Secret named `GOOGLE_API_KEY` and enable notebook access; the secret is then available as an environment variable. Alternatively, simply run the key cell below and paste the key into the hidden prompt. Never paste a key into a code cell as text: notebooks are shared, exported and committed, and a leaked key must be revoked.

In [ ]:
# Install the Gemini SDK. In Colab this takes a few seconds; if the package is
# already present, pip skips it. The "-q" flag keeps the output short.
%pip install -q google-generativeai

In [ ]:
import os
import json
from getpass import getpass
from typing import Any, Dict, List, Optional


def load_secret_from_environment(env_name: str, ask_if_missing: bool = False) -> Optional[str]:
    """Load a secret from an environment variable without printing it.

    This is the same safe-configuration pattern introduced in M01A. The secret
    stays outside the notebook source, and the function never echoes the value.
    If the variable is missing and ask_if_missing is True, the user is prompted
    once through getpass; pressing Enter without typing anything keeps the
    notebook in offline mode.
    """
    value = os.environ.get(env_name)
    if value:
        return value
    if ask_if_missing:
        typed = getpass(f"Enter {env_name} (press Enter to work offline): ")
        if typed:
            os.environ[env_name] = typed
            return typed
    return None


GOOGLE_API_KEY = load_secret_from_environment("GOOGLE_API_KEY", ask_if_missing=True)

# LIVE_MODE controls every model call in this notebook.
# True  -> prompts are sent to the Gemini API.
# False -> prompts are answered from recorded example outputs (offline mode).
LIVE_MODE = GOOGLE_API_KEY is not None

print("Live model mode:", LIVE_MODE)

The cell above prints only whether live mode is enabled, never the key itself. If you see `Live model mode: False`, the notebook will run entirely from recorded outputs, and every later cell still works. If you add a key later, re-run the cell above and the client cell below; you do not need to restart the runtime.

The next cell defines the offline recordings. Each entry pairs a *marker* (a distinctive phrase that appears in exactly one of the prompts used later in this lab) with a response recorded from a real instruction-tuned model. This lookup-by-marker design is intentionally simple: it is not a model, it is a tape recording, and its only job is to let you follow the lab without a key. When you write your own prompts in the student tasks, offline mode will return a clearly labelled placeholder instead of pretending to answer.

In [ ]:
# Recorded example outputs for offline mode.
# Each entry is (marker, recorded_response). The marker is a phrase that occurs
# in exactly one prompt used in this notebook, so the lookup is unambiguous.
# Order matters: the first matching marker wins, so more specific markers
# (such as the flipped-label demonstration) are listed before general ones.
MOCK_RESPONSES: List[tuple] = [
    # Section 4.1 - completion-style framing (no instruction given).
    ("teaches students how to",
     "build and evaluate generative AI systems, moving from prompt design and "
     "retrieval-augmented generation to tool-using agents, multi-agent "
     "coordination and safety testing. Students work in Python notebooks and "
     "visual workflow tools, and every project ends with an evaluation report."),

    # Section 4.1 - instruction-style framing over the same description.
    ("list exactly three topics",
     "- Prompt engineering and retrieval-augmented generation\n"
     "- Tool-using and multi-agent systems\n"
     "- Safety testing and evaluation of AI workflows"),

    # Section 4.3 - conflicting (flipped) demonstrations. Listed BEFORE the
    # normal demonstrations so the flipped marker is matched first.
    ('ran smoothly." -> negative',
     "negative"),

    # Section 4.3 - one-shot and few-shot with consistent demonstrations.
    # Listed BEFORE the Section 4.2 contract marker, because the 4.3 prompts
    # also contain the contract sentence and must match this entry first.
    ("easy to follow.",
     "positive"),

    # Section 4.2 - full specification with an output contract.
    ("exactly one lowercase word",
     "mixed"),

    # Section 4.2 - same task without an output contract (chatty answer).
    ("Classify the following student feedback comment",
     "This comment starts positively because the student found the examples "
     "clear, but it also raises a real usability problem with the final "
     "exercise. Overall I would describe it as a mixed review, leaning "
     "slightly positive."),
]

DEFAULT_MOCK_RESPONSE = (
    "[offline mode] No recorded response matches this prompt. Supply a "
    "GOOGLE_API_KEY for live answers, or add your own entry to MOCK_RESPONSES."
)


def mock_generate(prompt: str) -> str:
    """Return a recorded response whose marker appears in the prompt.

    One special case first: the zero-shot prompt in Section 4.3 contains the
    query comment but no Examples block. The recorded zero-shot answer drifted
    in format ("Positive." instead of "positive"), which is itself a teaching
    point: demonstrations standardise output format as well as meaning.
    """
    if "easy to follow." in prompt and "Examples:" not in prompt:
        return "Positive."
    for marker, response in MOCK_RESPONSES:
        if marker in prompt:
            return response
    return DEFAULT_MOCK_RESPONSE


print("Recorded responses loaded:", len(MOCK_RESPONSES))

In [ ]:
# Model client and the single entry point for all model calls in this lab.
MODEL_NAME = "gemini-2.5-flash"   # a fast, low-cost model; adjust if your
                                  # account offers a newer flash-class model.

model = None
if LIVE_MODE:
    import google.generativeai as genai
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel(MODEL_NAME)


def generate_text(prompt: Any, temperature: float = 0.2) -> Dict[str, Any]:
    """Send a prompt to the model, or answer from recordings in offline mode.

    The function returns the structured ok/error/result dictionary used
    throughout this unit, plus a "mode" field so that every experiment records
    whether its evidence came from a live model or from a recording. A low
    default temperature keeps live outputs relatively stable, which matters
    when we compare prompt variants later.
    """
    if not isinstance(prompt, str) or not prompt.strip():
        return {"ok": False, "error": "Prompt must be a non-empty string.",
                "result": None, "mode": "live" if LIVE_MODE else "mock"}

    if LIVE_MODE:
        try:
            response = model.generate_content(
                prompt,
                generation_config={"temperature": temperature},
            )
            return {"ok": True, "error": None,
                    "result": response.text.strip(), "mode": "live"}
        except Exception as exc:  # network, quota or safety-block errors
            return {"ok": False, "error": f"API call failed: {exc}",
                    "result": None, "mode": "live"}

    return {"ok": True, "error": None,
            "result": mock_generate(prompt), "mode": "mock"}


smoke_test = generate_text("Reply with the single word: ready")
print("Mode:", smoke_test["mode"])
print("OK:", smoke_test["ok"])

The smoke test confirms that the helper runs end to end. In live mode you should see `Mode: live` and `OK: True`; if you instead see an error mentioning authentication, your key was not loaded, and an error mentioning quota means the free-tier limit was reached (wait a minute and re-run). In offline mode you will see `Mode: mock`, and because the smoke-test prompt has no recorded marker, its result would be the labelled placeholder text, which is exactly the honest behaviour we want from a recording.

Note the design decision: every experiment in this lab calls `generate_text` and nothing else. Centralising model access in one function means validation, error handling and the live/offline switch live in one place. This is the same discipline you will apply when models become one component inside larger agentic workflows.

<a id="m02a-core-concepts"></a>

### 3. Core Concepts

**Model-level control versus runtime control.** Everything that shapes a model response comes from one of two places. Model-level control is baked in during training and is identical for every call you make. *Pretraining* on a large text corpus gives the model its knowledge, vocabulary and writing style. *Instruction tuning* teaches it to treat input as a request to satisfy rather than text to continue; this is the difference between a base model, which completes text, and the assistant-style models you meet through APIs. *Alignment* training shapes which requests the model declines or reframes, such as requests for harmful content. Runtime control is everything you decide per call: the prompt wording, the context you paste in, and, in retrieval-augmented systems, the evidence a retriever selects into that context.

```text
        MODEL-LEVEL CONTROL  (fixed during training, same for every call)
   +----------------------------------------------------------------+
   |  Pretraining         -> what the model knows, how it writes    |
   |  Instruction tuning  -> input is treated as a task request     |
   |  Alignment           -> which requests it refuses or reshapes  |
   +----------------------------------------------------------------+
                                 |
                          frozen weights
                                 |
        RUNTIME CONTROL  (chosen by you, can differ on every call)
   +----------------------------------------------------------------+
   |  Prompt     -> instruction, constraints, output contract       |
   |  Context    -> documents, examples, conversation history       |
   |  Retrieval  -> which evidence gets selected into the context   |
   +----------------------------------------------------------------+
                                 |
                                 v
                        one model response
```

The distinction matters for debugging. If a model ignores your formatting request, that is usually a runtime problem you can fix by rewriting the prompt. If a model refuses a legitimate request, that is alignment behaviour; no amount of clever rephrasing is the sanctioned fix, and attempting to bypass it is exactly the prompt-injection behaviour that `M06B` teaches you to defend against. If a model simply does not know a fact, that is a pretraining limitation, and the runtime remedy is to supply the fact in context, which is the entire motivation for RAG in [M02C](M02C-Retrieval-Augmented-Generation.ipynb).

**Prompts as specifications.** Because the prompt is your main runtime lever, treat it as an engineering artefact with named parts rather than as a sentence you improvise. A well-specified prompt answers five questions: what is the task, what material must be used, what constraints apply, what do good examples look like, and what exact shape must the output take. A role or persona is a sixth, optional field: include it only when a real audience or perspective changes the expected answer (an explanation "for first-year students" is a meaningful role effect; "you are a world-class genius" is not a specification, it is a superstition).

```text
   +---------------------------------------------------------------+
   |                  PROMPT AS A SPECIFICATION                    |
   |                                                               |
   |  Task             what the model must do                      |
   |  Context          the material it must work from              |
   |  Constraints      length, scope, tone, what to avoid          |
   |  Examples         demonstrations of input -> output           |
   |  Output contract  the exact, checkable shape of the answer    |
   |  (Role)           only when audience or perspective matters   |
   +---------------------------------------------------------------+
```

The *output contract* deserves special attention. "Respond with exactly one lowercase word: positive, negative, or mixed" is a contract because a short Python function can check compliance. "Keep it brief" is not a contract because no function can decide whether it was honoured. Measurable contracts are what turn prompting from an art into an engineering loop, and they are the foundation of the next session, [M02B](M02B-Prompt-Engineering-Control-Loop.ipynb).

**In-context learning.** An instruction-tuned model can perform tasks it was never explicitly trained on by inferring the task from the prompt alone. With no demonstrations (zero-shot) it relies on the instruction; with one demonstration (one-shot) it copies the demonstrated convention; with several (few-shot) the convention becomes more stable, covering label vocabulary, output format and edge-case handling. Crucially, the model weights do not change during in-context learning: the "learning" lives entirely inside the prompt and vanishes on the next call. This makes demonstrations powerful and dangerous in equal measure, because the model follows the convention the demonstrations establish, whether or not that convention is the one you intended. The conflicting-demonstration experiment in Section 4 makes this concrete.

<a id="m02a-guided-implementation"></a>

### 4. Guided Implementation

The guided work has three experiments, each isolating one idea. First, you contrast completion-style framing with instruction-style framing to see what instruction tuning changed. Second, you build prompts from explicit specification fields and observe what happens when a field is removed. Third, you run the same classification task under zero-, one- and few-shot demonstrations, and then under deliberately conflicting demonstrations.

All three experiments use a small, fully public piece of text: a short description of this unit, consistent with the repository data policy of preferring public unit material over external documents.

**Experiment 4.1: completion framing versus instruction framing.** Hosted APIs expose instruction-tuned models, not raw base models, so we cannot query a true base model here. We can, however, approximate the contrast at runtime: a prompt that ends mid-sentence invites the model to *continue* text (base-model-like behaviour that pretraining alone would produce), while a prompt that states a task invites it to *satisfy a request* (behaviour that exists because of instruction tuning). Watch which differences in the two outputs you can attribute to the runtime framing you chose, and which depend on the model having been instruction-tuned at all.

In [ ]:
# A short public unit description used as the working text for this lab.
unit_description = (
    "FLIP: Agentic AI in Practice teaches students how to "
    "design, build and evaluate generative and agentic AI systems. "
    "The unit covers prompt engineering, retrieval-augmented generation, "
    "visual workflows, LangChain and LangGraph programming, multi-agent "
    "collaboration, safety testing and model adaptation, using Python "
    "notebooks and public unit materials."
)

# Completion framing: the prompt is an unfinished sentence, not a request.
# A pure base model could only ever do this kind of continuation.
completion_prompt = "FLIP: Agentic AI in Practice teaches students how to"

# Instruction framing: the same subject matter, but stated as a task.
# Following this at all is instruction-tuned behaviour.
instruction_prompt = (
    "Read the unit description below and list exactly three topics it "
    "covers, one per line, each line starting with '- '.\n\n"
    f"Unit description:\n{unit_description}"
)

completion_result = generate_text(completion_prompt)
instruction_result = generate_text(instruction_prompt)

print("--- Completion framing ---")
print(completion_result["result"])
print()
print("--- Instruction framing ---")
print(instruction_result["result"])

Under completion framing, the output reads like a continuation of the sentence: fluent prose that extends the description, with no guaranteed structure. Under instruction framing, the output is a three-line list because that is what was requested. Now attribute the differences. The *existence* of request-following behaviour is model-level: instruction tuning put it there, and no prompt of yours created it. The *choice* between continuation and a three-item list is runtime-level: you selected it by changing only the framing, with the same frozen weights answering both calls. If you are in live mode, you may also notice the instruction-framed output is more consistent across re-runs than the completion-framed output; instructions plus contracts reduce variance, which is why specifications matter.

**Experiment 4.2: prompts as specifications.** Rather than writing prompts as free-form strings, we now build them from named fields. Programmatic construction has two benefits: every prompt documents which fields it contains, and you can vary exactly one field between runs, which is the controlled-comparison habit that `M02B` turns into a full methodology.

In [ ]:
def build_prompt(task: str,
                 context: Optional[str] = None,
                 constraints: Optional[List[str]] = None,
                 examples: Optional[List[tuple]] = None,
                 output_contract: Optional[str] = None,
                 role: Optional[str] = None) -> str:
    """Assemble a prompt from explicit specification fields.

    Each field is optional except the task, so experiments can remove one
    field at a time and observe the effect. Fields are labelled and separated
    by blank lines so both the model and a human reviewer can see the
    structure of the specification at a glance.
    """
    if not isinstance(task, str) or not task.strip():
        raise ValueError("A prompt specification requires a non-empty task.")

    parts: List[str] = []
    if role:
        parts.append(f"Role: {role}")
    parts.append(f"Task: {task}")
    if context:
        parts.append(f"Context:\n{context}")
    if constraints:
        bullet_list = "\n".join(f"- {c}" for c in constraints)
        parts.append(f"Constraints:\n{bullet_list}")
    if examples:
        demo_lines = "\n".join(f'"{text}" -> {label}' for text, label in examples)
        parts.append(f"Examples:\n{demo_lines}")
    if output_contract:
        parts.append(f"Output contract: {output_contract}")
    return "\n\n".join(parts)


# The classification micro-task used for the rest of this lab: label short
# student feedback comments about a lab session.
feedback_comment = ("The examples were clear, but the last exercise assumed "
                    "steps that were never shown.")

full_spec_prompt = build_prompt(
    task="Classify the following student feedback comment by sentiment.",
    context=f"Feedback comment: {feedback_comment}",
    constraints=[
        "Judge only the sentiment expressed in the comment itself.",
        "Do not explain your reasoning.",
    ],
    output_contract=("Respond with exactly one lowercase word: "
                     "positive, negative, or mixed."),
)

print(full_spec_prompt)

Read the printed prompt from top to bottom and notice that it answers every specification question: the task names the job, the context carries the material, the constraints bound the behaviour, and the output contract fixes the shape of the answer. There is no role field, deliberately: sentiment classification does not change with audience, so a persona would add words without adding control. The next cell runs this full specification, then runs the *same task with the output contract removed*, changing nothing else. This one-field-at-a-time comparison is the fair way to measure what a field contributes.

In [ ]:
# Variant with the output contract removed. Every other field is identical,
# so any behaviour difference is attributable to the missing contract.
no_contract_prompt = build_prompt(
    task="Classify the following student feedback comment by sentiment.",
    context=f"Feedback comment: {feedback_comment}",
    constraints=[
        "Judge only the sentiment expressed in the comment itself.",
        "Do not explain your reasoning.",
    ],
    output_contract=None,
)

with_contract = generate_text(full_spec_prompt)
without_contract = generate_text(no_contract_prompt)

print("--- With output contract ---")
print(with_contract["result"])
print()
print("--- Without output contract ---")
print(without_contract["result"])

With the contract, the answer is a single checkable word such as `mixed`. Without it, the model still understands the task but chooses its own format: typically a short paragraph that discusses both the positive and negative aspects before settling on a verdict, or sometimes a capitalised label with extra commentary. Neither output is *wrong* as English; the difference is that only the first can be verified by code and passed safely to the next step of a workflow. Whenever a model output will be consumed by a program rather than a person, an output contract is not optional politeness, it is the interface definition.

Because contracts are only useful if you check them, we now write the checker. It is small on purpose: a contract that needs a complicated checker is usually a badly designed contract.

In [ ]:
ALLOWED_LABELS = {"positive", "negative", "mixed"}


def check_label(model_output: Any) -> Dict[str, Any]:
    """Check a model response against the one-word sentiment contract.

    The checker is strict about substance but tolerant about trivia: it strips
    whitespace and a trailing full stop and lowercases the text, because those
    variations do not change meaning. Anything else - extra words, unknown
    labels, empty output, non-string input - is a contract violation. The
    structured ok/error/result shape matches the rest of the unit.
    """
    if not isinstance(model_output, str) or not model_output.strip():
        return {"ok": False, "error": "Output is empty or not a string.",
                "result": None}

    cleaned = model_output.strip().rstrip(".").lower()

    if cleaned in ALLOWED_LABELS:
        return {"ok": True, "error": None, "result": cleaned}

    if any(label in cleaned for label in ALLOWED_LABELS):
        return {"ok": False,
                "error": f"A label is present but wrapped in extra text: {model_output!r}",
                "result": None}

    return {"ok": False, "error": f"No allowed label found in: {model_output!r}",
            "result": None}


print("Contracted output :", check_label(with_contract["result"]))
print("Uncontracted output:", check_label(without_contract["result"]))

The contracted output passes the checker and yields a clean machine-readable label. The uncontracted output fails, and the error message tells you *how* it failed: usually a label buried inside extra text. That error category is worth remembering, because it is the most common real-world contract violation and it is invisible if you only eyeball outputs.

**Experiment 4.3: in-context learning with demonstrations.** We keep the same classification task and vary only the demonstrations. The zero-shot version relies on the instruction and contract alone; the one-shot and few-shot versions prepend worked examples through the `examples` field of `build_prompt`. The query comment is deliberately easy and clearly positive, so any change in the answer must come from the demonstrations, not from ambiguity in the input.

In [ ]:
# Demonstrations for the sentiment task. Each pair is (comment, label).
demonstrations = [
    ("The setup instructions were clear and everything ran smoothly.", "positive"),
    ("I wasted an hour because the notebook crashed on the second cell.", "negative"),
    ("Good coverage of concepts, although the pacing was uneven.", "mixed"),
]

# The query is intentionally unambiguous: a clearly positive comment.
query_comment = "The retrieval demo worked first time and the notes were easy to follow."

TASK = "Classify the following student feedback comment by sentiment."
CONTRACT = "Respond with exactly one lowercase word: positive, negative, or mixed."


def classify(comment: str, demos: Optional[List[tuple]] = None) -> Dict[str, Any]:
    """Run one classification call and check its contract in a single step.

    Bundling the model call and the checker keeps every experiment honest:
    each run records the raw output, the checked label and the mode, so a
    results table can be built without re-running anything.
    """
    prompt = build_prompt(
        task=TASK,
        context=f"Feedback comment: {comment}",
        examples=demos,
        output_contract=CONTRACT,
    )
    response = generate_text(prompt)
    checked = check_label(response["result"]) if response["ok"] else response
    return {"raw": response["result"], "checked": checked, "mode": response["mode"]}


zero_shot = classify(query_comment, demos=None)
one_shot = classify(query_comment, demos=demonstrations[:1])
few_shot = classify(query_comment, demos=demonstrations)

for name, run in [("zero-shot", zero_shot), ("one-shot", one_shot), ("few-shot", few_shot)]:
    print(f"{name:10s} raw={run['raw']!r:14} contract_ok={run['checked']['ok']} "
          f"label={run['checked']['result']}")

All three regimes should identify the comment as positive, because the task is easy. Look instead at the *format*. In the recorded offline run, the zero-shot answer came back as `'Positive.'` (capitalised, with a full stop) and failed the strict contract until the checker's normalisation rescued it, while the one-shot and few-shot answers came back as the exact lowercase word. This is the quiet, second job of demonstrations: they standardise output format, not just meaning. On harder tasks the demonstrations also carry the decision boundary, for example showing the model where you draw the line between `negative` and `mixed`.

Now the key experiment. We flip the labels in the demonstrations so that they contradict both common sense and the words used in the labels: clearly positive comments are demonstrated as `negative` and vice versa. The instruction and the query stay identical. If demonstrations were mere decoration, the answer would not change.

In [ ]:
# Conflicting demonstrations: identical comments, deliberately flipped labels.
flipped_demonstrations = [
    ("The setup instructions were clear and everything ran smoothly.", "negative"),
    ("I wasted an hour because the notebook crashed on the second cell.", "positive"),
]

conflicting = classify(query_comment, demos=flipped_demonstrations)

print("Query comment:", query_comment)
print("Demonstrated convention: positive comments -> 'negative', and vice versa")
print()
print(f"conflicting raw={conflicting['raw']!r} "
      f"contract_ok={conflicting['checked']['ok']} "
      f"label={conflicting['checked']['result']}")

In the recorded run, and in most live runs, the model answers `negative` for a clearly positive comment. It has inferred the mapping *from the demonstrations* rather than from the everyday meaning of the label words: the demonstrations established a convention, and the model followed it. Live models do not do this every time; when the conflict is strong, some runs revert to the common-sense label, and that instability is itself evidence that you have handed the model two contradictory sources of truth.

Sit with the implication for a moment, because it is the deepest lesson of this session. The output was fluent, confident and contract-compliant, and it was also wrong by your intent. Nothing in the response signals the problem; you can only detect it because you know what the demonstrations said. In later modules, retrieval systems and agent workflows will assemble context automatically, and a retrieval bug that inserts misleading examples or stale documents produces exactly this failure: a well-formed answer built on bad context. This is why `M02C` insists on tracing answers back to evidence, and why `M06B` treats context as an attack surface.

<a id="m02a-testing"></a>

### 5. Testing and Analysis

Testing a prompt-driven system has two distinct layers, and this section deliberately keeps them apart. The *plumbing* - `build_prompt`, `check_label`, the input validation inside `generate_text` - is deterministic code, so we test it with hard `assert` statements exactly as in M01. The *model behaviour* is probabilistic in live mode, so hard asserts on it would make the notebook fail randomly; instead we run behavioural checks that record and report outcomes. Learning which layer a given check belongs to is part of the lab.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Test type</strong></th>
<th><strong>Layer</strong></th>
<th><strong>Example in this notebook</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Normal case</td><td>Plumbing (assert)</td><td>A full specification produces a prompt containing every labelled field; a clean label passes <code>check_label</code>.</td></tr>
<tr><td align="left">Edge case</td><td>Plumbing (assert)</td><td>A minimal task-only specification still builds; <code>"Positive."</code> normalises to <code>positive</code>.</td></tr>
<tr><td align="left">Failure case</td><td>Plumbing (assert)</td><td>Empty prompts are rejected by <code>generate_text</code>; a chatty answer fails the contract; an empty task raises <code>ValueError</code>.</td></tr>
<tr><td align="left">Normal case</td><td>Behaviour (report)</td><td>Few-shot classification of a clear comment returns a contract-compliant label.</td></tr>
<tr><td align="left">Edge case</td><td>Behaviour (report)</td><td>A genuinely two-sided comment should resolve to <code>mixed</code>.</td></tr>
<tr><td align="left">Failure case</td><td>Behaviour (report)</td><td>Conflicting demonstrations produce a contract-compliant but semantically flipped label - a silent failure.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# ---- Plumbing tests: deterministic, so hard asserts are appropriate. ----

# Normal case: a full specification contains every labelled field in order.
built = build_prompt(
    task="Classify the comment.",
    context="Feedback comment: example",
    constraints=["Be concise."],
    examples=[("good lab", "positive")],
    output_contract="One word.",
)
assert "Task: Classify the comment." in built
assert "Context:" in built and "Constraints:" in built
assert "Examples:" in built and "Output contract:" in built

# Edge case: a task-only specification is minimal but valid.
minimal = build_prompt(task="Say hello.")
assert minimal == "Task: Say hello."

# Edge case: harmless format drift is normalised, not rejected.
drift = check_label("Positive.")
assert drift["ok"] is True and drift["result"] == "positive"

# Failure case: a label buried in extra text violates the contract.
chatty = check_label("I would say this is mixed, leaning positive.")
assert chatty["ok"] is False and chatty["result"] is None

# Failure case: empty and non-string outputs are rejected safely.
assert check_label("")["ok"] is False
assert check_label(None)["ok"] is False

# Failure case: generate_text rejects empty or non-string prompts
# before any network call is attempted.
assert generate_text("")["ok"] is False
assert generate_text(42)["ok"] is False

# Failure case: a specification without a task is a programming error.
try:
    build_prompt(task="   ")
    raise AssertionError("build_prompt should reject an empty task.")
except ValueError:
    pass

print("All plumbing tests passed.")

If the cell prints `All plumbing tests passed.`, the deterministic layer honours its specification: prompts are assembled with every requested field, minimal input still works, and every invalid input is rejected with a structured error instead of a crash. As in M01, passing tests are evidence about the behaviours you specified, not a proof of general correctness.

The behavioural checks below run three classification cases and print a small results table. In offline mode the table is fully reproducible. In live mode, expect the normal case to pass essentially always, the edge case to pass most of the time (a model may defensibly call a two-sided comment `negative`), and the conflicting case to *pass the contract while failing your intent* - which is why the table reports the semantic expectation separately from contract compliance.

In [ ]:
# ---- Behavioural checks: probabilistic in live mode, so report, do not assert. ----

behaviour_cases = [
    {
        "name": "normal: clear positive, few-shot",
        "comment": query_comment,
        "demos": demonstrations,
        "expected": "positive",
    },
    {
        "name": "edge: genuinely two-sided comment",
        "comment": ("Good coverage of concepts, although the pacing was uneven."),
        "demos": demonstrations,
        "expected": "mixed",
    },
    {
        "name": "failure: conflicting demonstrations",
        "comment": query_comment,
        "demos": flipped_demonstrations,
        "expected": "positive",   # our intent - the flipped demos fight it
    },
]

print(f"{'case':42s} {'contract':9s} {'label':10s} {'matches intent'}")
print("-" * 78)
for case in behaviour_cases:
    run = classify(case["comment"], demos=case["demos"])
    label = run["checked"]["result"]
    contract_ok = run["checked"]["ok"]
    matches = (label == case["expected"])
    print(f"{case['name']:42s} {str(contract_ok):9s} {str(label):10s} {matches}")

print()
print("Note: 'contract' checks output shape; 'matches intent' checks meaning.")
print("The conflicting case is designed to satisfy the first and fail the second.")

The table separates two judgements that beginners often conflate. *Contract compliance* asks whether the output has the right shape, and code can check it. *Intent match* asks whether the output means what you wanted, and checking it requires ground truth that the output alone does not carry. The conflicting-demonstration row typically shows `contract=True` with `matches intent=False`: a syntactically perfect, semantically wrong answer. Real evaluation of prompt-driven systems always needs both layers, and `M02B` builds the full loop - target, intervention, observation, evaluation, diagnosis - around exactly this distinction.

One more analysis habit before the student tasks: whenever a behavioural check fails, ask *which control layer* the failure lives in. A wrong label with honest demonstrations is a specification or model-capability issue; a wrong label with corrupted demonstrations is a context issue; a refusal is an alignment issue. Locating the layer tells you which lever can fix it.

<a id="m02a-student-tasks"></a>

### 6. Student Tasks

The guided sections used a sentiment micro-task chosen by us. You now repeat the methodology on a task you design, so that the habits - full specification, one-field-at-a-time comparison, demonstration experiments, layered checking - transfer beyond this notebook. Choose a small text-labelling or text-extraction task with a closed set of valid outputs (for example: classifying unit announcements as `deadline`, `content` or `admin`; or extracting the module code such as `M02` from a sentence). A closed output set is required because Task 2 asks you to write a checker for it.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Task 1</td><td>Define your micro-task and write a full specification prompt for it with <code>build_prompt</code>, using task, context, constraints and an output contract (add a role only if you can justify it in one sentence).</td><td>Practises writing prompts as checkable specifications rather than improvised requests.</td><td>The printed prompt showing every field, plus your one-sentence role justification or the statement that no role is needed.</td></tr>
<tr><td align="left">Task 2</td><td>Write <code>check_my_contract(model_output)</code> for your output contract, following the <code>check_label</code> pattern, and test it: a normal case (valid label passes), an edge case (harmless format drift is normalised), and failure cases (extra text, unknown label, empty string and non-string input all return <code>ok=False</code> without raising).</td><td>A contract you cannot check by code is not a contract; the checker is also your grading tool for Task 3.</td><td>The checker function and passing assert output for normal, edge and failure cases.</td></tr>
<tr><td align="left">Task 3</td><td>Run your task zero-shot, one-shot and few-shot (at least three demonstrations) on the same three input cases: one normal, one edge (genuinely ambiguous), and one failure probe using conflicting demonstrations. Record results in a table like Section 5.</td><td>Shows how demonstrations change accuracy and format stability, and reproduces the silent-failure pattern on your own task.</td><td>A results table covering all regime-by-case combinations, with contract compliance and intent match reported separately.</td></tr>
<tr><td align="left">Task 4</td><td>Write a short analysis (100 to 200 words) attributing at least two observed behaviours to model-level control and at least two to runtime control.</td><td>Checks that you can locate behaviour in the correct control layer, which is the diagnostic skill the whole session builds.</td><td>The written analysis in a markdown cell.</td></tr>
</tbody>
</table>

</div>

For the programming work in Tasks 2 and 3, the expected behaviours are:

<div align="center">

<table>
<thead>
<tr>
<th><strong>Input case</strong></th>
<th><strong>Example</strong></th>
<th><strong>Expected behaviour</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Normal case</td><td>A clear input from your task with few-shot demonstrations</td><td>Contract-compliant output that matches your intended label.</td></tr>
<tr><td align="left">Edge case</td><td>A genuinely ambiguous input</td><td>Contract-compliant output; record which label the model chose and whether you agree.</td></tr>
<tr><td align="left">Failure case</td><td>Conflicting demonstrations with a clear input</td><td>Output may satisfy the contract while contradicting your intent; your table must flag it.</td></tr>
<tr><td align="left">Failure case</td><td><code>check_my_contract("")</code> and <code>check_my_contract(None)</code></td><td>Return <code>ok=False</code> with a clear error message, never raise an exception.</td></tr>
</tbody>
</table>

</div>

If you are working offline, `generate_text` will return the labelled placeholder for your new prompts. In that case, add two or three recorded entries to `MOCK_RESPONSES` for your own prompts (write plausible model outputs by hand and mark them as hand-written in a comment), or complete Task 3 by reasoning about expected outputs and stating clearly that they are predictions. Either route is acceptable; silently presenting placeholder text as model output is not.

In [ ]:
# Student task starter.
# Task 1: define your micro-task and build its full specification prompt.

# TODO: describe your task and its closed output set here.
# MY_ALLOWED_OUTPUTS = {...}

# TODO: build your full specification prompt.
# my_prompt = build_prompt(
#     task=...,
#     context=...,
#     constraints=[...],
#     output_contract=...,
# )
# print(my_prompt)

# Task 2: write your contract checker following the check_label pattern.
# def check_my_contract(model_output: Any) -> Dict[str, Any]:
#     ...

In [ ]:
# Student task tests.
# Uncomment and adapt after completing Tasks 1 and 2.

# Normal case: a valid output passes.
# ok_case = check_my_contract("your-valid-label")
# assert ok_case["ok"] is True

# Edge case: harmless drift is normalised.
# drift_case = check_my_contract("Your-Valid-Label.")
# assert drift_case["ok"] is True

# Failure cases: rejected safely, never raising.
# assert check_my_contract("some longer explanation text")["ok"] is False
# assert check_my_contract("")["ok"] is False
# assert check_my_contract(None)["ok"] is False

# print("Student contract checker tests passed.")

# Task 3: run your zero/one/few-shot and conflicting-demonstration
# experiments here, and print a results table like the one in Section 5.

<a id="m02a-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence. The emphasis is on controlled comparison and layered checking, not on any single model answer being correct.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Required item</strong></th>
<th><strong>What to submit</strong></th>
<th><strong>Quality check</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Specification prompt</td><td>Your Task 1 prompt built with <code>build_prompt</code>, printed in full.</td><td>Contains task, context, constraints and an output contract; role included only with a written justification.</td></tr>
<tr><td align="left">Contract checker</td><td><code>check_my_contract</code> and its tests.</td><td>Normal, edge and failure asserts all pass; invalid input never raises.</td></tr>
<tr><td align="left">Demonstration experiment</td><td>The Task 3 results table.</td><td>Covers zero/one/few-shot on normal, edge and conflicting cases; reports contract compliance and intent match as separate columns; states whether results are live, recorded or predicted.</td></tr>
<tr><td align="left">Control-layer analysis</td><td>The Task 4 written analysis.</td><td>At least two behaviours correctly attributed to each control layer.</td></tr>
<tr><td align="left">Reflection</td><td>150 to 250 words.</td><td>Refers to concrete observations from your own runs, not generic statements.</td></tr>
</tbody>
</table>

</div>

Reflection questions:

1. Which behaviours in your experiments could you change at runtime, and which were fixed by training?
2. Why is an output contract more useful than an instruction such as "keep it brief"?
3. What did the conflicting-demonstration experiment show about where a model's task convention comes from?
4. Why is a contract-compliant but semantically wrong answer more dangerous than a malformed one?
5. How would the silent-failure pattern you observed appear in a RAG system whose retriever returns misleading documents?

Use the debugging guide below if the notebook does not behave as expected.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Symptom</strong></th>
<th><strong>Likely cause</strong></th>
<th><strong>How to inspect</strong></th>
<th><strong>Typical fix</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left"><code>Live model mode: False</code> unexpectedly</td><td>Key not present in the environment</td><td>Re-run the key cell and check for typos in <code>GOOGLE_API_KEY</code></td><td>Set the Colab Secret or paste the key into the hidden prompt, then re-run the client cell</td></tr>
<tr><td align="left"><code>API call failed</code> mentioning quota or 429</td><td>Free-tier rate limit reached</td><td>Read the error text in the returned dictionary</td><td>Wait a minute and re-run; run experiments one cell at a time</td></tr>
<tr><td align="left"><code>API call failed</code> mentioning the model name</td><td><code>MODEL_NAME</code> not available on your account</td><td>List available models in AI Studio</td><td>Change <code>MODEL_NAME</code> to a flash-class model you can access</td></tr>
<tr><td align="left">Offline placeholder text in results</td><td>Your new prompt has no recorded marker</td><td>Check whether the output starts with <code>[offline mode]</code></td><td>Add a <code>MOCK_RESPONSES</code> entry for your prompt or supply a key</td></tr>
<tr><td align="left">Contract checks fail on live outputs</td><td>Model wrapped the label in extra words</td><td>Print the raw output next to the checker error</td><td>Tighten the contract wording; add one demonstration of a bare label</td></tr>
<tr><td align="left">Conflicting case matches intent</td><td>Live model overrode the flipped demos</td><td>Re-run a few times and count outcomes</td><td>This is a legitimate observation - report the instability</td></tr>
</tbody>
</table>

</div>

When you have completed the submission items, continue to [M02B: Prompt Engineering as a Control Loop](M02B-Prompt-Engineering-Control-Loop.ipynb), which turns the one-field-at-a-time habit from this session into a full iterative methodology.

#### Further Readings

- Google Gemini prompting strategies: <https://ai.google.dev/gemini-api/docs/prompting-strategies>
- Google AI Studio (API keys and model access): <https://aistudio.google.com>
- OpenAI prompt engineering guide: <https://platform.openai.com/docs/guides/prompt-engineering>
- Brown et al., "Language Models are Few-Shot Learners" (GPT-3, in-context learning): <https://arxiv.org/abs/2005.14165>
- Ouyang et al., "Training language models to follow instructions with human feedback" (instruction tuning and alignment): <https://arxiv.org/abs/2203.02155>
- Min et al., "Rethinking the Role of Demonstrations" (what demonstrations actually teach): <https://arxiv.org/abs/2202.12837>